# 📊 พยากรณ์ปริมาณการฆ่าสัตว์และประเมินกำลังการผลิตเนื้อสัตว์รายจังหวัด (Animal Slaughter Volume & Capacity Prediction)
## คลาสเรียนรู้ Machine Learning สำหรับการวิเคราะห์ความสัมพันธ์และพยากรณ์ปริมาณการแปรรูปปศุสัตว์
### 📊 แสดงผลแผนภูมิด้วย Plotly (Interactive Chart) เพื่อความสวยงามระดับ Premium และรองรับภาษาไทย 100%

---

### 📋 วัตถุประสงค์
1. รวมข้อมูลการปศุสัตว์รายจังหวัด (จำนวนประชากรสัตว์ และ จำนวนครัวเรือนเกษตรกร) ร่วมกับสถิติการอนุญาตให้ฆ่าสัตว์เพื่อการบริโภค
2. เรียนรู้การเตรียมข้อมูล (Data Preprocessing) และแก้ไขปัญหาข้อมูลไม่สมดุล/มีความเบ้สูง (Data Imbalance / Skewness) ด้วยวิธี **Target Log-Transformation**
3. สร้างและเปรียบเทียบโมเดล Machine Learning (Linear Regression & Gradient Boosting Regressor) เพื่อทำนายปริมาณการแปรรูปเนื้อสัตว์
4. วิเคราะห์ตัวแปรสำคัญ (Feature Importance) เพื่อระบุปัจจัยเชิงพื้นที่ที่ขับเคลื่อนการบริโภคและการผลิตเนื้อสัตว์ในแต่ละภูมิภาค

## 1. Setup & Import Libraries 📦

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# ตั้งค่า renderer สำหรับ VS Code Jupyter Notebook ให้แสดงผลกราฟิกแบบตอบสนองได้
pio.renderers.default = 'notebook_connected'
pio.templates.default = 'plotly_white'

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print("✅ Setup สำเร็จ และพร้อมใช้งาน (ใช้ Plotly สำหรับการวาดกราฟ)")

✅ Setup สำเร็จ และพร้อมใช้งาน (ใช้ Plotly สำหรับการวาดกราฟ)


## 2. Load and Clean Datasets 📁
ทำการโหลดข้อมูลจำนวนสถิติการฆ่าสัตว์ และข้อมูลจำนวนประชากรสัตว์/เกษตรกรรายจังหวัด

In [2]:
# ฟังก์ชันทำความสะอาดข้อมูลตามกฎที่ตั้งไว้ใน EDA
def clean_and_load_csv(filepath):
    df = pd.read_csv(filepath, encoding='utf-8-sig')
    df.columns = [col.strip() for col in df.columns]
    
    def clean_cell(val):
        if pd.isna(val):
            return np.nan
        if isinstance(val, str):
            val_str = val.strip()
            if val_str == '-':
                return np.nan
            cleaned_num = val_str.replace(',', '')
            try:
                return float(cleaned_num)
            except ValueError:
                return val_str
        return val
        
    for col in df.columns:
        df[col] = df[col].apply(clean_cell)
    return df

# โหลดข้อมูล
data_dir = "Animal"
df_kill = clean_and_load_csv(os.path.join(data_dir, "animal_can_kill.csv"))

df_pig = clean_and_load_csv(os.path.join(data_dir, "total_pig.csv"))
df_cowmeat = clean_and_load_csv(os.path.join(data_dir, "total_cowmeat.csv"))
df_cow = clean_and_load_csv(os.path.join(data_dir, "total_cow.csv")) # กระบือ

df_fpig = clean_and_load_csv(os.path.join(data_dir, "total_farmer_pig.csv"))
df_fcowmeat = clean_and_load_csv(os.path.join(data_dir, "total_farmer_cowmeat.csv"))
df_fcow = clean_and_load_csv(os.path.join(data_dir, "total_farmer_cow.csv")) # เกษตรกรเลี้ยงกระบือ

print("📊 โหลดข้อมูลปศุสัตว์เสร็จสมบูรณ์!")
print(f" - ขนาดข้อมูลสถิติการฆ่า: {df_kill.shape}")
print(f" - ขนาดข้อมูลประชากรสุกร: {df_pig.shape}")

📊 โหลดข้อมูลปศุสัตว์เสร็จสมบูรณ์!
 - ขนาดข้อมูลสถิติการฆ่า: (693, 5)
 - ขนาดข้อมูลประชากรสุกร: (385, 8)


## 3. Merge Datasets & Feature Engineering 🔗
เชื่อมโยงข้อมูลปริมาณประชากร เกษตรกร และข้อมูลสถิติการแปรรูปเนื้อสัตว์รายจังหวัดเข้าด้วยกัน

In [3]:
# เตรียมเชื่อมโยงข้อมูลประชากรและเกษตรกร
pig_data = df_pig[['year', 'province', 'value', 'region']].rename(columns={'value': 'pig_pop'})
cattle_data = df_cowmeat[['year', 'province', 'value']].rename(columns={'value': 'cattle_pop'})
buffalo_data = df_cow[['year', 'province', 'value']].rename(columns={'value': 'buffalo_pop'})

fpig_data = df_fpig[['year', 'province', 'value']].rename(columns={'value': 'pig_farmers'})
fcattle_data = df_fcowmeat[['year', 'province', 'value']].rename(columns={'value': 'cattle_farmers'})
fbuffalo_data = df_fcow[['year', 'province', 'value']].rename(columns={'value': 'buffalo_farmers'})

# เปลี่ยนชื่อคอลัมน์ของข้อมูลการฆ่าสัตว์ให้เข้ากันได้
kill_data = df_kill.rename(columns={
    'ปี': 'year', 
    'จังหวัด': 'province',
    'โค': 'slaughter_cow',
    'กระบือ': 'slaughter_buffalo',
    'สุกร': 'slaughter_pig'
})

# ทำการ Merge ข้อมูลเข้าด้วยกันทีละขั้น
df_merged = kill_data.copy()
for df_temp in [pig_data, cattle_data, buffalo_data, fpig_data, fcattle_data, fbuffalo_data]:
    df_merged = pd.merge(df_merged, df_temp, on=['year', 'province'], how='inner')

# จัดการค่าว่าง (NaN) ในสถิติการฆ่าสัตว์ให้เป็น 0 (หมายถึงไม่มีการฆ่าเพื่อบริโภคอย่างเป็นทางการในพื้นที่นั้น)
df_merged['slaughter_cow'] = df_merged['slaughter_cow'].fillna(0)
df_merged['slaughter_buffalo'] = df_merged['slaughter_buffalo'].fillna(0)
df_merged['slaughter_pig'] = df_merged['slaughter_pig'].fillna(0)

print(f"📈 ข้อมูลรวมพร้อมสำหรับสร้างโมเดล! มีจำนวนแถวทั้งหมด: {len(df_merged)} แถว (ครอบคลุม 5 ปี 77 จังหวัด)")
display(df_merged.head(10))

📈 ข้อมูลรวมพร้อมสำหรับสร้างโมเดล! มีจำนวนแถวทั้งหมด: 385 แถว (ครอบคลุม 5 ปี 77 จังหวัด)


,year,province,slaughter_cow,slaughter_buffalo,slaughter_pig,pig_pop,region,cattle_pop,buffalo_pop,pig_farmers,cattle_farmers,buffalo_farmers
0,2564,กรุงเทพมหานคร,1275.0,10.0,1070.0,2843,ภาคกลาง,4050,276,6,571,47
1,2564,นนทบุรี,2906.0,0.0,0.0,0,ภาคกลาง,1934,164,0,293,35
2,2564,ปทุมธานี,38729.0,0.0,426287.0,8129,ภาคกลาง,4517,682,61,272,58
3,2564,พระนครศรีอยุธยา,868.0,0.0,10185.0,2537,ภาคกลาง,10330,1387,63,1072,164
4,2564,อ่างทอง,0.0,0.0,64055.0,70391,ภาคกลาง,13346,766,848,1557,79
5,2564,ลพบุรี,5489.0,0.0,478186.0,520745,ภาคกลาง,56548,3377,1958,3605,221
6,2564,สิงห์บุรี,120.0,0.0,17183.0,42923,ภาคกลาง,2879,117,301,420,24
7,2564,ชัยนาท,695.0,115.0,45960.0,160026,ภาคกลาง,49569,16416,1172,2923,1192
8,2564,สระบุรี,2719.0,0.0,317887.0,167669,ภาคกลาง,30292,10982,251,2014,660
9,2564,สมุทรปราการ,0.0,0.0,24960.0,91,ภาคกลาง,280,32,4,35,6


## 4. Train-Test Split 🕒
เพื่อหลีกเลี่ยงการรั่วไหลของข้อมูลเวลา (Data Leakage) เราจะไม่สุ่มแยกข้อมูล แต่จะแบ่งตามปีงบประมาณ:
- **Train Set**: ข้อมูลปี 2564 ถึง 2567 (4 ปีแรก)
- **Test Set**: ข้อมูลปีล่าสุด 2568 (เพื่อทดสอบการทำนายอนาคต)

In [4]:
# คัดเลือก Features และ Targets
features = ['pig_pop', 'cattle_pop', 'buffalo_pop', 'pig_farmers', 'cattle_farmers', 'buffalo_farmers']
targets = ['slaughter_pig', 'slaughter_cow', 'slaughter_buffalo']

# แบ่ง Train / Test ตามแกนเวลา
train_mask = df_merged['year'] < 2568
test_mask = df_merged['year'] == 2568

df_train = df_merged[train_mask]
df_test = df_merged[test_mask]

X_train, y_train = df_train[features], df_train[targets]
X_test, y_test = df_test[features], df_test[targets]

print(f"📂 ข้อมูล Train (ปี 2564-2567): {len(X_train)} แถว")
print(f"📂 ข้อมูล Test (ปี 2568): {len(X_test)} แถว")

📂 ข้อมูล Train (ปี 2564-2567): 308 แถว
📂 ข้อมูล Test (ปี 2568): 77 แถว


## 5. Model Training & Evaluation 🤖
### 🧠 การแก้ไขปัญหา Data Imbalance / Skewness
เนื่องจากข้อมูลสถิติการฆ่าสัตว์มีความเบ้สูงมาก (บางจังหวัดเลี้ยงและฆ่าสัตว์ปริมาณหลายแสนตัว ในขณะที่ส่วนใหญ่เป็นศูนย์หรือน้อยมาก) โมเดลปกติจะถูกดึงประสิทธิภาพโดยกลุ่มข้อมูลที่เป็น Outliers เหล่านี้

เราจึงใช้วิธี **Target Log-Transformation** ในการแปลงข้อมูลเป้าหมายให้อยู่ในรูป $y' = \log(y + 1)$ ก่อนนำไปฝึกสอนโมเดล และแปลงผลลัพธ์กลับด้วย $y = \exp(y') - 1$ ในขั้นตอนประเมินผล

และเปรียบเทียบการทำนายระหว่าง **Linear Regression** และ **Gradient Boosting Regressor** (โดยตั้งค่า `loss='huber'` เพื่อช่วยป้องกันอิทธิพลของข้อมูลที่เบ้จัด)

In [5]:
results = {}

for target in targets:
    print(f"\n{'='*20} 🎯 ทำนายเป้าหมาย: {target} {'='*20}")
    
    # แปลงข้อมูลเป้าหมายด้วย Log scale
    y_train_log = np.log1p(y_train[target])
    
    # 1. Linear Regression (บน Log scale)
    lr = LinearRegression()
    lr.fit(X_train, y_train_log)
    lr_pred_log = lr.predict(X_test)
    lr_pred = np.expm1(lr_pred_log)
    lr_pred = np.clip(lr_pred, 0, None)  # ป้องกันค่าการทำนายติดลบ
    
    # 2. Gradient Boosting Regressor (บน Log scale + Huber Loss)
    gbr = GradientBoostingRegressor(loss='huber', n_estimators=100, random_state=42)
    gbr.fit(X_train, y_train_log)
    gbr_pred_log = gbr.predict(X_test)
    gbr_pred = np.expm1(gbr_pred_log)
    gbr_pred = np.clip(gbr_pred, 0, None)
    
    # วัดค่าประสิทธิภาพบนสเกลจริง (Original Scale)
    lr_r2 = r2_score(y_test[target], lr_pred)
    lr_mae = mean_absolute_error(y_test[target], lr_pred)
    lr_rmse = np.sqrt(mean_squared_error(y_test[target], lr_pred))
    
    gbr_r2 = r2_score(y_test[target], gbr_pred)
    gbr_mae = mean_absolute_error(y_test[target], gbr_pred)
    gbr_rmse = np.sqrt(mean_squared_error(y_test[target], gbr_pred))
    
    print(f"[Linear Regression (Log)] R2: {lr_r2:.4f} | MAE: {lr_mae:,.2f} | RMSE: {lr_rmse:,.2f}")
    print(f"[Gradient Boosting (Log)] R2: {gbr_r2:.4f} | MAE: {gbr_mae:,.2f} | RMSE: {gbr_rmse:,.2f}")
    
    results[target] = {
        'lr_pred': lr_pred,
        'gbr_pred': gbr_pred,
        'y_actual': y_test[target].values,
        'gbr_model': gbr,
        'metrics': {
            'lr': {'R2': lr_r2, 'MAE': lr_mae, 'RMSE': lr_rmse},
            'gbr': {'R2': gbr_r2, 'MAE': gbr_mae, 'RMSE': gbr_rmse}
        }
    }


==================== 🎯 ทำนายเป้าหมาย: slaughter_pig ====================


[Linear Regression (Log)] R2: -0.0781 | MAE: 217,551.06 | RMSE: 742,484.20


[Gradient Boosting (Log)] R2: 0.7067 | MAE: 127,540.56 | RMSE: 387,272.18

==================== 🎯 ทำนายเป้าหมาย: slaughter_cow ====================


[Linear Regression (Log)] R2: -1.3317 | MAE: 4,231.78 | RMSE: 9,020.15


[Gradient Boosting (Log)] R2: 0.4749 | MAE: 2,177.92 | RMSE: 4,280.61

==================== 🎯 ทำนายเป้าหมาย: slaughter_buffalo ====================


[Linear Regression (Log)] R2: -19.0406 | MAE: 1,318.56 | RMSE: 5,975.69
[Gradient Boosting (Log)] R2: 0.5218 | MAE: 360.43 | RMSE: 923.08


## 6. Visualization 📊
สร้างกราฟ Plotly แบบพรีเมียม เพื่อแสดงผลการเปรียบเทียบและความสำคัญของแต่ละปัจจัย

### 6.1 แผนภูมิตัวแปรสำคัญ (Feature Importance)
วิเคราะห์ปัจจัยประชากรและเกษตรกรสัตว์ประเภทใดที่มีผลต่อกำลังผลิตแปรรูปของแต่ละเป้าหมาย

In [6]:
# แสดง Feature Importance ของโมเดล Gradient Boosting สำหรับเป้าหมายแต่ละตัว
for target in targets:
    gbr_model = results[target]['gbr_model']
    importances = gbr_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    imp_df = pd.DataFrame({
        'Feature': [features[i] for i in indices],
        'Importance': importances[indices]
    })
    
    # แสดงผลด้วยแผนภูมิแท่งแนวนอนโดยใช้โทนสีพรีเมียมเข้มแบบไม่มีส่วนผสมของสีขาว
    fig = px.bar(
        imp_df.sort_values('Importance', ascending=True), 
        y='Feature', 
        x='Importance', 
        orientation='h',
        title=f'ปัจจัยสำคัญในการทำนายผลผลิต {target} (Gradient Boosting)',
        color='Importance',
        color_continuous_scale=['#9575CD', '#311B92'] # โทนสีม่วงคมชัดตัดขอบสวยงาม
    )
    fig.update_layout(
        yaxis_title='ตัวแปรต้น/ปัจจัย',
        xaxis_title='ค่าน้ำหนักความสำคัญ (Relative Importance)',
        coloraxis_showscale=False,
        height=400,
        title_x=0.5,
        font=dict(size=12)
    )
    fig.show()

### 6.2 กราฟเปรียบเทียบค่าจริงกับการทำนาย (Actual vs. Predicted)
เปรียบเทียบผลทำนายของโมเดล Gradient Boosting ในจังหวัดที่สถิติจริงสูงสุด 20 ลำดับแรกในปี 2568

In [7]:
# วาดกราฟเปรียบเทียบ Actual vs Predicted ของสุกร (ซึ่งมีข้อมูลหลากหลายและชัดเจนที่สุด)
target_show = 'slaughter_pig'
test_provinces = df_test['province'].values
actual = results[target_show]['y_actual']
pred = results[target_show]['gbr_pred']

compare_df = pd.DataFrame({
    'Province': test_provinces,
    'Actual': actual,
    'Predicted': pred
}).sort_values(by='Actual', ascending=False).head(20) # แสดง 20 จังหวัดแรกที่มีปริมาณสูงสุด

fig = go.Figure()
fig.add_trace(go.Bar(
    name='จำนวนผลิตจริง (Actual)',
    x=compare_df['Province'],
    y=compare_df['Actual'],
    marker_color='#34495e'
))
fig.add_trace(go.Bar(
    name='จำนวนทำนาย (Predicted)',
    x=compare_df['Province'],
    y=compare_df['Predicted'],
    marker_color='#e74c3c'
))

fig.update_layout(
    title=f'เปรียบเทียบสถิติจริงและคำทำนายโมเดล (Actual vs. Predicted) ปี 2568 - {target_show}',
    xaxis_title='จังหวัด',
    yaxis_title='จำนวนสัตว์แปรรูป (ตัว)',
    barmode='group',
    title_x=0.5,
    legend=dict(x=0.8, y=0.9),
    height=500
)
fig.show()

### 6.3 กราฟประเมินค่า R2 Score ของโมเดล
เปรียบเทียบประสิทธิภาพ R2 Score ระหว่างโมเดลเพื่อสรุปผลผลิตเนื้อสัตว์ทั้ง 3 ประเภท

In [8]:
metrics_data = []
for target in targets:
    metrics_data.append({
        'Animal': target.replace('slaughter_', '').upper(),
        'Model': 'Linear Regression (Log)',
        'R2': results[target]['metrics']['lr']['R2']
    })
    metrics_data.append({
        'Animal': target.replace('slaughter_', '').upper(),
        'Model': 'Gradient Boosting (Log)',
        'R2': results[target]['metrics']['gbr']['R2']
    })
df_metrics = pd.DataFrame(metrics_data)

fig = px.bar(
    df_metrics, 
    x='Animal', 
    y='R2', 
    color='Model', 
    barmode='group',
    title='เปรียบเทียบประสิทธิภาพ R2 Score ระหว่างโมเดล (ประเมินบนสเกลจริง)',
    color_discrete_map={'Linear Regression (Log)': '#bdc3c7', 'Gradient Boosting (Log)': '#9b59b6'}
)
fig.update_layout(
    yaxis_title='R2 Score (ความแม่นยำเชิงสัมพัทธ์)',
    xaxis_title='ประเภทสัตว์',
    title_x=0.5,
    height=450
)
fig.show()